# 03 — Build and verify initial Gold

Prepare operational facts, delivery KPIs and separate acceptance-time training/scoring datasets.
Requires wheel **aidp_lakehouse_to_ai-0.3.0-py3-none-any.whl**, authenticated AIDP Spark/Delta compute,
Standard catalog lakehouse_demo, existing gold schema, read access to five managed Silver tables
and v2 control files, and Gold CREATE TABLE / insert / query permissions. Install and restart first.
Run top to bottom in a fresh session; no variables or functions from other notebooks are required.

**User-reported live Silver:** 600 customers / 30,600 orders / 30,000 deliveries / 94,572 events;
zero rejects, empty quarantine; rerun inserted zero rows and preserved fingerprints.
The agent did not execute that AIDP run. **Live Gold verification: pending.**

Expected Gold: 30,600 operational rows, 30,000 training rows, 600 scoring rows. KPI count depends
on observed UTC delivery-month / acceptance-region / actual-carrier groups. An exactly-on-promise
completion is on time; open operational labels are NULL. Scoring has no label or probability.
See docs/aidp-gold-setup.md. No training, inference, agents, Autonomous writes or incremental processing.

In [ ]:
# The only environment configuration section. No credentials.
CONFIG = {
    "catalog": "lakehouse_demo",
    "namespace": "frgpslcqn7sm",
    "bucket": "CristinaVaras-bucket-AIDP",
    "prefix": "demos/lakehouse-to-ai/source-data/v2/seed-42/reference-2026-10-28",
    "reference_date": "2026-10-28",
    "seed": 42,
    "batch_id": "initial",
}

## 1. Runtime, configuration and explicit contracts

In [ ]:
import json
try:
    from aidp_lakehouse import __version__
    from aidp_lakehouse.aidp_bronze import BronzeConfig
    from aidp_lakehouse.gold import GoldConfig, VERSION, CONTRACTS_GOLD, KEYS, PREDICTORS, IDENTIFIERS, LABEL, TRAINING_METADATA
    from aidp_lakehouse.gold_spark import build_initial_gold
    if __version__ != VERSION:
        raise RuntimeError(f"Install wheel {VERSION} and restart the session")
    cfg = GoldConfig(BronzeConfig(**CONFIG))
    spark.conf.set("spark.sql.session.timeZone", "UTC")
    print("Spark:", spark.version, "Package:", __version__)
    print("Configuration:", json.dumps(CONFIG, sort_keys=True))
    print("Identifiers:", IDENTIFIERS)
    print("Acceptance-time predictors:", PREDICTORS)
    print("Training label:", LABEL, "Metadata:", TRAINING_METADATA)
    print("Contracts:", json.dumps(CONTRACTS_GOLD, indent=2))
except Exception as error:
    print(f"GOLD FAIL [runtime/configuration]: {error}")
    raise

## 2. Validate Silver, transform and publish initial Gold

The reusable module pins Silver Delta versions, confirms managed Delta schemas, initial-v2 provenance,
counts, empty quarantine, keys and lifecycle relationships. Joins and aggregations are distributed.
Events are aggregated before joining. All outputs and existing targets are preflighted before any write.
Compatible initial rows are preserved; only missing rows are inserted. Incompatible definitions or
changed/extra rows stop the build. Use one writer: commits are per table, not atomic across all four.

In [ ]:
try:
    first = build_initial_gold(spark, cfg)
    print(json.dumps(first, indent=2, default=str))
except Exception as error:
    print(f"GOLD FAIL [initial build]: {error}")
    raise

## 3. Verify an independent rerun

Re-read Silver and rebuild. Require identical validation summaries/fingerprints and zero new target rows.
Full-row multiset equality against published tables is authoritative; fingerprints alone are not proof.

In [ ]:
try:
    second = build_initial_gold(spark, cfg)
    if first["validation"] != second["validation"] or first["fingerprints"] != second["fingerprints"]:
        raise RuntimeError("Gold rerun changed business results")
    if any(row["inserted_rows"] != 0 for row in second["publication"].values()):
        raise RuntimeError("Gold rerun unexpectedly inserted rows")
    print(json.dumps(second, indent=2, default=str))
except Exception as error:
    print(f"GOLD FAIL [rerun]: {error}")
    raise

## 4. Small samples and final result

In [ ]:
try:
    for entity in CONTRACTS_GOLD:
        print("Gold sample:", entity)
        spark.table(f"{cfg.gold_namespace}.{entity}").orderBy(*KEYS[entity]).limit(3).show(truncate=False)
    print("GOLD PASS: keys, cardinalities, labels/nullability, KPIs and identical rerun; zero second-run inserts.")
    print("Data preparation only. Training, scoring and agents are future milestones.")
except Exception as error:
    print(f"GOLD FAIL [final verification]: {error}")
    raise